In [1]:
import os
from hydra import compose, initialize
from utils.demo_utils import *

### Import model

Load configuration using Hydra and initialize the pre-trained model.

In [2]:
with initialize(version_base=None, config_path="configs"):
    cfg = compose(config_name="base")
predictor = Predictor(cfg, weights_path='./weights.pt')

### Process model input

Here, we process the PDB file (e.g., 3eo1.pdb) to extract the antigen and select hotspot residues. Follow these steps:

1. Set the path to your PDB file containing the antigen structure.
2. Run `create_antigen_pdb` and `visualize_antigen_with_selection` to extract and display the antigen.
3. Click on desired epitope regions in the visualization; note the unique residue numbers shown.
4. Update `hotspot_residues` with those numbers (e.g., key binding sites like [138, 143, 144, 204]).
5. Run `process_model_input` to prepare the batch data using selected hotspots, antibody PDB, and CDR loop lengths.

In [3]:
pdb_path = './demo/inputs/3eo1.pdb' # <-- Adjust here

create_antigen_pdb(pdb_path)
visualize_antigen_with_selection(pdb_path.replace('.pdb', '_antigen.pdb'))

Found 2 antigen chain(s): ['A', 'B']
Antigen-only PDB saved to: ./demo/inputs/3eo1_antigen.pdb


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [4]:
hotspot_residues = [142, 143, 144, 204, 205, 206] # <-- Adjust here
cdr_loop_lengths = {'H1':7, 'H2':6, 'H3':(5,21), 'L1':(8,13), 'L2':7, 'L3':(9,11)} # <-- Adjust here

antibody_pdb = './demo/inputs/hu-4D5-8_Fv.pdb'
antigen_pdb = pdb_path.replace('.pdb', '_antigen.pdb')

### Run inference and evaluate results

Run prediction on GPU; generated PDB files are saved to the `sample_dir` you defined.  

Each generated sample is further evaluated using the following metrics to assess the designed antibody:
- Hotspot Conservation (HC): The ratio of successfully covered (Recall) or correctly predicted (Precision) hotspot residues.
- Min CDR-Antigen Distance: The minimum tip-to-tip distance between any CDR loop and the antigen surface, indicating the closest point of interaction.
- Framework RMSD: Structural deviation of framework Cα atoms from the ground truth.
- Num of clashing residues: Total number of steric clashes at the backbone level.

In [5]:
sample_dir = './demo/outputs/3eo1_epitope1/' # <-- Adjust here
_inference_num_per_sample = 1 # <-- Adjust here

for bidx in range(_inference_num_per_sample):
    batch_, sample_ranges = process_model_input(
            antigen_pdb=antigen_pdb,
            hotspot_residues=hotspot_residues,
            antibody_pdb=antibody_pdb,
            cdr_loop_lengths=cdr_loop_lengths
        )

    print(f"Generating sample {bidx} ...")
    predictor.predict(batch_, bidx, sample_dir, sample_ranges)

    sample_path = os.path.join(sample_dir, f'sample_{bidx}.pdb')

    comment_parts = [f"C{res}" for res in hotspot_residues]
    comment_line = f"REMARK EPITOPE-LABEL: {', '.join(comment_parts)}\n"
    with open(sample_path, 'a') as f:
        f.writelines(comment_line)

    quantitative_evaluation(batch_, sample_path)

Generating sample 0 ...


100%|██████████| 99/99 [00:17<00:00,  5.59it/s]


Minimum antibody-antigen distance: 5.63 Å
Hotspot Precision: 0.6000, Recall: 1.0000
Framework RMSD: 0.1623189359350761
Number of Clashing Residues: 0 at []


### Vizualize results

Visualize the generated sample PDB at the specified sample_path (e.g., ./demo_outputs/3eo1/sample_0.pdb).
- Dark gray: Antigen
- Red: Epitope (hotspots)
- Beige: Antibody
- Yellow: CDR loops

In [6]:
sample_path = './demo/outputs/3eo1_epitope1/sample_0.pdb' # <-- Adjust here
visualize_output(sample_path)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.